# Étape 2 : modèles GPU (TabPFN et TabM)

Ce notebook calcule les prédictions des modèles qui demandent un GPU et les écrit au même format que
`1_modeles_cpu.py` : un fichier `predictions/<modèle>.npz` par modèle, avec
- `hors_pli` : les 84 000 lignes de développement, chaque pli prédit par un modèle entraîné sur les 4 autres ;
- `test_interne` : les 21 000 lignes mises de côté, modèle entraîné sur les 84 000 lignes ;
- `test` : les 45 000 lignes de `test.csv`, modèle entraîné sur les 105 000 lignes.

| Modèle | Rôle | Durée indicative (GPU T4) |
|---|---|---|
| `tabpfn` | TabPFN 3.5, 8 passages, cible ramenée au poids (solution finale) | environ 3 h |
| `tabm`, `tabm_allom` | les deux TabM de la solution finale | environ 30 min |
| `tabpfn_ancien` | TabPFN, 4 passages, cible en kg (comparaisons seulement) | environ 1 h |

**Avant de lancer :** *Exécution → Modifier le type d'exécution → GPU*, et un secret Colab `TABPFN_TOKEN`
(clé d'API Prior Labs ; la licence de TabPFN doit être acceptée sur ux.priorlabs.ai).

**Ensuite :** *Exécution → Tout exécuter*. Chaque morceau terminé est sauvegardé sur Google Drive : si Colab
se coupe, relancer *Tout exécuter* reprend où le calcul s'était arrêté. À la fin, `predictions_gpu.zip`
est téléchargé : en extraire les fichiers dans le dossier `predictions/` du dépôt, puis lancer
`python 3_solution_finale.py`.

Les calculs GPU ne sont pas bit à bit reproductibles d'une carte à l'autre : les prédictions peuvent
différer de celles fournies de quelques centièmes de kilogramme (TabPFN) à quelques dixièmes (TabM).

In [ ]:
import os, sys, json, time, subprocess, importlib.metadata
from pathlib import Path
from google.colab import userdata, drive, files

# Clé d'API Prior Labs (licence TabPFN acceptée sur ux.priorlabs.ai), rangée dans les secrets Colab.
jeton = userdata.get('TABPFN_TOKEN')
assert isinstance(jeton, str) and jeton.strip(), 'Le secret TABPFN_TOKEN est absent ou vide.'
os.environ['TABPFN_TOKEN'] = jeton.strip(); os.environ['TABPFN_NO_BROWSER'] = '1'; del jeton

# Versions de TabPFN et TabM du projet ; on garde torch, numpy, pandas, scipy et scikit-learn de Colab.
BASE = ['torch', 'numpy', 'pandas', 'scipy', 'scikit-learn']
Path('/content/contraintes.txt').write_text('\n'.join(f'{n}=={importlib.metadata.version(n)}' for n in BASE) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==9.1.0', 'tabm==0.0.3', 'rtdl_num_embeddings==0.0.12',
                'lightgbm', 'catboost', '--constraint', '/content/contraintes.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Pas de GPU : Exécution → Modifier le type d’exécution → GPU.'
print('GPU :', torch.cuda.get_device_name(0))

# Le dépôt contient le code et les données.
if not Path('/content/depot').is_dir():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Hamdane11/performances-powerlifting', '/content/depot'], check=True)
sys.path.insert(0, '/content/depot/src')
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from powerlifting.data import charger, configuration, decoupage
from powerlifting.models import predict_model

CFG = configuration()
train, test = charger()
d = decoupage(len(train))
X_all, y_all = train.drop(columns=['TotalKg']), train['TotalKg'].to_numpy(float)
print(len(d.dev), 'lignes de développement,', len(d.test_interne), 'de test interne,', len(test), 'de test')

COLONNES = list(CFG['variables'])

def preparer(frame):
    x = frame.loc[:, COLONNES].copy().reset_index(drop=True)
    x.loc[x['Age'] < 5, 'Age'] = np.nan
    for c in COLONNES[3:]:
        x[c] = x[c].fillna('Unknown').astype(str)
    return x

def tabpfn_8_passages(contexte, y_contexte, requetes, passages=8, exposant=2/3, bloc=15000):
    """TabPFN 3.5 : 8 passages, cible divisée par (poids/80)^(2/3), espérance de la loi prédite."""
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    xc, xq = preparer(contexte), preparer(requetes)
    echelle = lambda x: (x['BodyweightKg'].to_numpy(float) / 80.0) ** exposant
    m = TabPFNRegressor.create_default_for_version(
        ModelVersion.V3_5, n_estimators=passages, device='cuda', random_state=2026,
        fit_mode='fit_preprocessors', memory_saving_mode=True, n_preprocessing_jobs=1,
        categorical_features_indices=list(range(3, 16)), inference_config={'SUBSAMPLE_SAMPLES': None},
        tuning_config=None, average_before_softmax=False, ignore_pretraining_limits=False)
    m.fit(xc, np.asarray(y_contexte, float) / echelle(xc))
    sq, p = echelle(xq), np.empty(len(xq))
    for a in range(0, len(xq), bloc):
        p[a:a + bloc] = np.asarray(m.predict(xq.iloc[a:a + bloc], output_type='mean'), float) * sq[a:a + bloc]
    del m; torch.cuda.empty_cache()
    return np.maximum(p, 0.0)

MODELES = ['tabpfn', 'tabm', 'tabm_allom']          # ajouter 'tabpfn_ancien' pour les comparaisons
SAUVEGARDE = Path('/content/drive/MyDrive/powerlifting/modeles_gpu')

def modele(nom):
    """Fonction (contexte, cibles, requêtes) -> prédictions pour chaque modèle GPU."""
    if nom == 'tabpfn':
        return tabpfn_8_passages
    famille = 'tabm' if nom.startswith('tabm') else 'tabpfn'
    return lambda c, y, q: predict_model(famille, c.reset_index(drop=True), y, q.reset_index(drop=True),
                                         dict(CFG['modeles'][nom]), COLONNES)

## Calcul (reprend là où il s'était arrêté)

In [ ]:
for nom in MODELES:
    dossier = SAUVEGARDE / nom
    dossier.mkdir(parents=True, exist_ok=True)
    taches = {f'pli{k}': (d.dev[d.plis != k], d.dev[d.plis == k]) for k in range(1, 6)}
    taches['test_interne'] = (d.dev, d.test_interne)
    taches['test'] = (np.arange(len(train)), None)
    for tache, (apprentissage, requetes) in taches.items():
        fichier = dossier / f'{tache}.npy'
        if fichier.is_file():
            continue
        t = time.time()
        q = test if requetes is None else X_all.iloc[requetes]
        np.save(fichier, modele(nom)(X_all.iloc[apprentissage], y_all[apprentissage], q))
        print(f'{nom} {tache} : {(time.time() - t) / 60:.1f} min', flush=True)
    hors_pli = np.empty(len(d.dev))
    for k in range(1, 6):
        hors_pli[d.plis == k] = np.load(dossier / f'pli{k}.npy')
    Path('/content/predictions').mkdir(exist_ok=True)
    np.savez_compressed(f'/content/predictions/{nom}.npz', hors_pli=hors_pli,
                        test_interne=np.load(dossier / 'test_interne.npy'), test=np.load(dossier / 'test.npy'))
    rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
    print(f'{nom} : RMSE hors pli {rmse(y_all[d.dev], hors_pli):.2f} kg, '
          f'test interne {rmse(y_all[d.test_interne], np.load(dossier / "test_interne.npy")):.2f} kg')

In [ ]:
import shutil
files.download(shutil.make_archive('/content/predictions_gpu', 'zip', '/content/predictions'))